# CourtListener Keyword vs. Semantic Search

This notebook runs the same case-name query twice through `CourtListenerClient.search`: once with the default keyword search and once with `semantic=True`. It displays the returned count, exact-name rank, ranking scores, citations, and the first 20 candidates.

Keep `CASE_NAME_QUERIES` short: each name makes two CourtListener API requests and the configured account rate limits still apply.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
from IPython.display import display

from mellea_lrc.core.env import load_env_file
from mellea_lrc.courtlistener import CourtListenerClient

load_env_file(Path("../.env"), override=True)

# Start with one query to stay comfortably inside CourtListener's rate limits.
CASE_NAME_QUERIES = ["Johnson v. City of Shelby"]
SEARCH_TYPE = "o"  # Case-law opinion clusters; semantic search only supports case law.

client = CourtListenerClient()
pd.set_option("display.max_colwidth", 100)

In [2]:
def normalize_case_name(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", value.casefold()).strip()


def search_rows(query: str, *, semantic: bool) -> list[dict[str, object]]:
    response = client.search(query, SEARCH_TYPE, semantic=semantic)
    raw = response["raw"]
    results = raw.get("results", []) if isinstance(raw, dict) else []
    total_count = raw.get("count") if isinstance(raw, dict) else None
    mode = "semantic" if semantic else "keyword"
    expected = normalize_case_name(query)

    rows = []
    for rank, result in enumerate(results, start=1):
        case_name = str(result.get("caseName") or result.get("case_name") or "")
        scores = result.get("meta", {}).get("score", {})
        rows.append(
            {
                "query": query,
                "mode": mode,
                "total_count": total_count,
                "rank": rank,
                "exact_name": normalize_case_name(case_name) == expected,
                "case_name": case_name,
                "court_id": result.get("court_id") or result.get("court"),
                "date_filed": result.get("dateFiled") or result.get("date_filed"),
                "citations": result.get("citation"),
                "bm25_score": scores.get("bm25"),
                "semantic_score": scores.get("semantic"),
                "absolute_url": result.get("absolute_url"),
            }
        )
    return rows

In [3]:
rows = []
for query in CASE_NAME_QUERIES:
    rows.extend(search_rows(query, semantic=False))
    rows.extend(search_rows(query, semantic=True))

results = pd.DataFrame(rows)

summary_rows = []
for (query, mode), group in results.groupby(["query", "mode"], sort=False):
    exact_ranks = group.loc[group["exact_name"], "rank"].tolist()
    summary_rows.append(
        {
            "query": query,
            "mode": mode,
            "reported_count": group["total_count"].iloc[0],
            "returned_results": len(group),
            "first_exact_rank": exact_ranks[0] if exact_ranks else None,
        }
    )

display(pd.DataFrame(summary_rows))
display(
    results[
        [
            "mode",
            "rank",
            "exact_name",
            "case_name",
            "court_id",
            "date_filed",
            "citations",
            "bm25_score",
            "semantic_score",
            "absolute_url",
        ]
    ]
)

,query,mode,reported_count,returned_results,first_exact_rank
0,Johnson v. City of Shelby,keyword,4754,20,1.0
1,Johnson v. City of Shelby,semantic,576,20,NaN


,mode,rank,exact_name,case_name,court_id,date_filed,citations,bm25_score,semantic_score,absolute_url
0,keyword,1,True,Johnson v. City of Shelby,scotus,2016-10-03,"[137 S. Ct. 146, 196 L. Ed. 2d 44, 41 I.E.R. Cas. (BNA) 1236, 85 U.S.L.W. 3139, 2016 U.S. LEXIS ...",4876.099600,NaN,/opinion/8427090/johnson-v-city-of-shelby/
1,keyword,2,True,Johnson v. City of Shelby,scotus,2014-11-10,"[135 S. Ct. 346, 190 L. Ed. 2d 309, 90 Fed. R. Serv. 3d 224, 25 Fla. L. Weekly Fed. S 5, 39 I.E....",4639.229000,NaN,/opinion/2750101/johnson-v-city-of-shelby/
2,keyword,3,False,Fagan v. Shelby,ohioctapp,2025-07-23,[2025 Ohio 2648],841.670840,NaN,/opinion/10642792/fagan-v-shelby/
3,keyword,4,False,City of Grants Pass v. Johnson,scotus,2024-06-28,[603 U.S. 520],779.927250,NaN,/opinion/10600044/city-of-grants-pass-v-johnson/
4,keyword,5,False,"City of Memphis v. Shelby County, Tennessee",tennctapp,2015-02-20,"[469 S.W.3d 531, 2015 Tenn. App. LEXIS 77, 2015 WL 739849]",774.873000,NaN,/opinion/2780837/city-of-memphis-v-shelby-county-tennessee/
5,keyword,6,False,Morton v. City of Shelby,missctapp,2007-11-20,"[984 So. 2d 323, 2007 WL 4108718]",702.878660,NaN,/opinion/1700571/morton-v-city-of-shelby/
6,keyword,7,False,"Surgeon v. TKO Shelby, LLC",nc,2024-03-22,[],696.535300,NaN,/opinion/9487043/surgeon-v-tko-shelby-llc/
7,keyword,8,False,City of Strongsville v. Johnson,ohioctapp,2017-08-03,"[2017 Ohio 7066, 95 N.E.3d 809]",690.166930,NaN,/opinion/4415150/city-of-strongsville-v-johnson/
8,keyword,9,False,Jones v. City of Seattle,wash,2013-12-12,"[179 Wash. 2d 322, 314 P.3d 380]",688.427550,NaN,/opinion/4909436/jones-v-city-of-seattle/
9,keyword,10,False,City of Dallas v. Arredondo,texapp,2013-08-13,"[415 S.W.3d 327, 2013 WL 4076868]",667.900630,NaN,/opinion/5286101/city-of-dallas-v-arredondo/
